# Regression: Predicting Semantic Drift Before Running Chains

Goal: build pre-experiment predictors for final semantic stability. The default target is step-10 word-vs-guess semantic similarity, averaged across instances for each `word ? model ? pipeline`. Exact accuracy is kept as a secondary target.

This notebook intentionally avoids using chain-behavior features from later steps as predictors. It uses word/category features, model/pipeline metadata, embedding geometry of the original words, and golden-description features.


In [ ]:
from pathlib import Path
import hashlib
import math
import re
import sqlite3

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'datasets').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'datasets').exists():
    raise FileNotFoundError('Could not find the project root containing datasets/.')
OUTPUT_DIR = PROJECT_ROOT / 'data_analysis' / 'results_analysis' / 'regression_model'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

UNIFIED_CSV = PROJECT_ROOT / 'datasets' / 'unified_semantic_drift_results.csv'
GOLDEN_DESCRIPTIONS_CSV = PROJECT_ROOT / 'datasets' / 'golden_descriptions_gpt_oss.csv'
WORD_CATEGORIES_XLSX = PROJECT_ROOT / 'datasets' / 'word_categories_latest.xlsx'
WORD_FREQ_CSV = PROJECT_ROOT / 'datasets' / 'words_with_usage_count.csv'
WORD_WEIRD_CSV = PROJECT_ROOT / 'datasets' / 'words_weird.csv'

# Existing cache from semantic-similarity analysis.
EMBEDDING_CACHE = (
    PROJECT_ROOT / 'data_analysis' / 'results_analysis' / 'all_guess_similarity_description_similarity'
    / 'cache' / 'embeddings_sentence_transformers_all_minilm_l6_v2.sqlite'
)
EMBEDDING_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'

SELECTED_PIPELINES = ['pipeline_a', 'pipeline_b']
TARGET_STEP = 10
NEIGHBOR_K = 10
RANDOM_STATE = 42

MODEL_META = {
    'Llama 3.1 8B Instruct': {'model_family': 'Llama', 'parameter_b': 8},
    'Llama 3.1 70B Instruct': {'model_family': 'Llama', 'parameter_b': 70},
    'Gemma 3 4B IT': {'model_family': 'Gemma', 'parameter_b': 4},
    'Gemma 3 12B IT': {'model_family': 'Gemma', 'parameter_b': 12},
    'Gemma 3 27B IT': {'model_family': 'Gemma', 'parameter_b': 27},
}

PIPELINE_FROM_PROMPT_MODE = {
    'pipeline_A_guess_then_describe_guessed_word': 'pipeline_a',
    'pipeline_B_paraphrase_description': 'pipeline_b',
}

CATEGORY_ORDER = [
    'High-Freq-Concrete',
    'Low-Freq-Concrete',
    'High-Freq-Abstract',
    'Low-Freq-Abstract',
]

print('Embedding cache exists:', EMBEDDING_CACHE.exists())


## Helpers


In [ ]:
TOKEN_RE = re.compile(r"[A-Za-z0-9]+")


def normalize_word(value):
    if pd.isna(value):
        return ''
    value = str(value).strip().lower()
    value = re.sub(r'[^a-zA-Z\s\-]', '', value)
    return value.strip()


def normalize_guess(value):
    if pd.isna(value):
        return ''
    value = str(value).strip().lower()
    value = re.sub(r'^the word is\s+', '', value)
    value = re.sub(r'^answer:\s*', '', value)
    value = re.sub(r'^guess:\s*', '', value)
    value = re.sub(r'[^a-zA-Z\s\-]', '', value)
    return value.strip()


def tokenize(text):
    return TOKEN_RE.findall(str(text).lower())


def parse_number(value):
    if pd.isna(value):
        return np.nan
    return pd.to_numeric(str(value).replace(',', '.'), errors='coerce')


def text_hash(text):
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


def load_cached_embeddings(texts, cache_path=EMBEDDING_CACHE, model_name=EMBEDDING_MODEL):
    texts = list(dict.fromkeys(str(text) for text in texts if isinstance(text, str) and text.strip()))
    embeddings = {}
    if not cache_path.exists():
        return embeddings, texts
    conn = sqlite3.connect(cache_path)
    try:
        for start in range(0, len(texts), 800):
            batch = texts[start:start + 800]
            hashes = [text_hash(text) for text in batch]
            hash_to_text = dict(zip(hashes, batch))
            placeholders = ','.join(['?'] * len(batch))
            rows = conn.execute(
                f'''
                SELECT text_hash, text, dim, vector
                FROM embeddings
                WHERE model_name = ? AND text_hash IN ({placeholders})
                ''',
                [model_name, *hashes],
            ).fetchall()
            for text_hash_value, text, dim, vector in rows:
                if hash_to_text.get(text_hash_value) == text:
                    embeddings[text] = np.frombuffer(vector, dtype=np.float32, count=dim)
    finally:
        conn.close()
    missing = [text for text in texts if text not in embeddings]
    return embeddings, missing


def cosine_from_embeddings(text_a, text_b, embeddings):
    vec_a = embeddings.get(text_a)
    vec_b = embeddings.get(text_b)
    if vec_a is None or vec_b is None:
        return np.nan
    return float(np.dot(vec_a, vec_b))


## Target Table

Primary target: `step10_semantic_similarity`, mean word-vs-guess semantic similarity at step 10 for each `model ? pipeline ? category ? word`.

Secondary target: `step10_exact_accuracy`.


In [ ]:
usecols = ['model_name', 'category', 'original_word', 'step', 'guess', 'prompt_mode']
results = pd.read_csv(UNIFIED_CSV, usecols=usecols, dtype='string')
results['pipeline'] = results['prompt_mode'].map(PIPELINE_FROM_PROMPT_MODE).fillna(results['prompt_mode'])
results['step'] = pd.to_numeric(results['step'], errors='coerce')
results = results[results['pipeline'].isin(SELECTED_PIPELINES) & results['step'].eq(TARGET_STEP)].copy()
results['word_norm'] = results['original_word'].map(normalize_word)
results['guess_norm'] = results['guess'].map(normalize_guess)
results['exact_correct'] = results['word_norm'].eq(results['guess_norm'])

texts = pd.concat([results['word_norm'], results['guess_norm']]).dropna().astype(str).unique().tolist()
embeddings, missing_target_texts = load_cached_embeddings(texts)
print(f'Loaded embeddings for {len(embeddings):,} texts; missing {len(missing_target_texts):,}.')

results['word_guess_semantic_similarity'] = [
    cosine_from_embeddings(word, guess, embeddings)
    for word, guess in zip(results['word_norm'], results['guess_norm'])
]

targets = (
    results
    .groupby(['model_name', 'pipeline', 'category', 'original_word'], observed=True)
    .agg(
        step10_semantic_similarity=('word_guess_semantic_similarity', 'mean'),
        step10_exact_accuracy=('exact_correct', 'mean'),
        correct_count=('exact_correct', 'sum'),
        row_n=('exact_correct', 'size'),
        semantic_valid_n=('word_guess_semantic_similarity', 'count'),
        unique_step10_guesses=('guess_norm', 'nunique'),
    )
    .reset_index()
)

targets['word_norm'] = targets['original_word'].map(normalize_word)
targets.to_csv(OUTPUT_DIR / 'regression_targets_step10.csv', index=False)
targets.head()


## Word and Lexical Features


In [ ]:
# Category membership from the target table is enough for the 400 words.
word_features = targets[['category', 'original_word', 'word_norm']].drop_duplicates().copy()
word_features['word_length_chars'] = word_features['word_norm'].str.replace(' ', '', regex=False).str.len()
word_features['word_token_count'] = word_features['word_norm'].map(lambda x: len(tokenize(x)))
word_features['is_high_frequency_category'] = word_features['category'].astype(str).str.startswith('High').astype(int)
word_features['is_concrete_category'] = word_features['category'].astype(str).str.endswith('Concrete').astype(int)

# SUBTLEX-style frequency counts.
if WORD_FREQ_CSV.exists():
    freq = pd.read_csv(WORD_FREQ_CSV, dtype='string')
    freq['word_norm'] = freq['Word'].map(normalize_word)
    for col in ['FREQcount', 'CDcount', 'FREQlow', 'Cdlow', 'SUBTLWF', 'Lg10WF', 'SUBTLCD', 'Lg10CD']:
        if col in freq.columns:
            freq[col] = freq[col].map(parse_number)
    keep_cols = ['word_norm'] + [col for col in ['FREQcount', 'CDcount', 'SUBTLWF', 'Lg10WF', 'SUBTLCD', 'Lg10CD'] if col in freq.columns]
    word_features = word_features.merge(freq[keep_cols].drop_duplicates('word_norm'), on='word_norm', how='left')

# Extra lexical columns: Zipf frequency and dominant POS if available.
if WORD_WEIRD_CSV.exists():
    weird = pd.read_csv(WORD_WEIRD_CSV, dtype='string')
    weird['word_norm'] = weird['Spelling'].map(normalize_word)
    for col in ['FreqCount', 'LogFreq(Zipf)', 'CD_count', 'CD']:
        if col in weird.columns:
            weird[col] = weird[col].map(parse_number)
    keep_cols = ['word_norm'] + [col for col in ['FreqCount', 'LogFreq(Zipf)', 'CD_count', 'CD', 'DomPoS'] if col in weird.columns]
    weird = weird[keep_cols].drop_duplicates('word_norm')
    word_features = word_features.merge(weird, on='word_norm', how='left', suffixes=('', '_weird'))

word_features.head()


## Embedding Geometry Features

These features use only the original target words, so they are available before running the chain experiment. Because the cached embeddings are normalized, `word_embedding_norm` will probably be near 1 for every word and may be dropped by the model-prep step if it has no variance.


In [ ]:
unique_words = sorted(word_features['word_norm'].dropna().unique())
word_embeddings, missing_word_embeddings = load_cached_embeddings(unique_words)
print(f'Word embeddings found: {len(word_embeddings):,}; missing: {len(missing_word_embeddings):,}')

matrix_words = [word for word in unique_words if word in word_embeddings]
embedding_matrix = np.vstack([word_embeddings[word] for word in matrix_words]) if matrix_words else np.empty((0, 0))
word_to_idx = {word: i for i, word in enumerate(matrix_words)}

embedding_rows = []
if len(matrix_words):
    sim_matrix = embedding_matrix @ embedding_matrix.T
    all_centroid = embedding_matrix.mean(axis=0)
    all_centroid = all_centroid / np.linalg.norm(all_centroid)

    category_centroids = {}
    for category, sub in word_features.groupby('category', observed=True):
        words = [word for word in sub['word_norm'] if word in word_to_idx]
        if not words:
            continue
        centroid = np.vstack([word_embeddings[word] for word in words]).mean(axis=0)
        centroid = centroid / np.linalg.norm(centroid)
        category_centroids[category] = centroid

    for word in matrix_words:
        idx = word_to_idx[word]
        row = word_features[word_features['word_norm'].eq(word)].iloc[0]
        similarities = np.delete(sim_matrix[idx], idx)
        top_k = np.sort(similarities)[-NEIGHBOR_K:] if len(similarities) >= NEIGHBOR_K else similarities
        vec = word_embeddings[word]
        category_centroid = category_centroids.get(row['category'])
        category_centroid_similarity = float(np.dot(vec, category_centroid)) if category_centroid is not None else np.nan
        embedding_rows.append({
            'word_norm': word,
            'word_embedding_norm': float(np.linalg.norm(vec)),
            f'neighbor_density_top{NEIGHBOR_K}': float(np.mean(top_k)) if len(top_k) else np.nan,
            'nearest_neighbor_similarity': float(np.max(similarities)) if len(similarities) else np.nan,
            'all_words_centroid_similarity': float(np.dot(vec, all_centroid)),
            'category_centroid_similarity': category_centroid_similarity,
            'category_centroid_distance': 1.0 - category_centroid_similarity if not np.isnan(category_centroid_similarity) else np.nan,
        })

embedding_features = pd.DataFrame(embedding_rows)
word_features = word_features.merge(embedding_features, on='word_norm', how='left')
word_features.head()


## Golden Description Features

These features are also available before running the model chains, because they come from the 100 initial descriptions per word.


In [ ]:
golden = pd.read_csv(GOLDEN_DESCRIPTIONS_CSV, dtype='string')
golden['word_norm'] = golden['Word'].map(normalize_word)
golden['description'] = golden['Description'].fillna('').astype(str)
golden['description_char_len'] = golden['description'].str.len()
golden['description_tokens'] = golden['description'].map(tokenize)
golden['description_token_count'] = golden['description_tokens'].map(len)
golden['description_type_count'] = golden['description_tokens'].map(lambda toks: len(set(toks)))
golden['description_ttr'] = golden.apply(
    lambda row: row['description_type_count'] / row['description_token_count'] if row['description_token_count'] else np.nan,
    axis=1,
)

# Lexical diversity over all 100 descriptions: unique tokens / total tokens.
def corpus_ttr(token_lists):
    tokens = [tok for toks in token_lists for tok in toks]
    return len(set(tokens)) / len(tokens) if tokens else np.nan

basic_desc_features = (
    golden
    .groupby('word_norm', observed=True)
    .agg(
        description_n=('description', 'size'),
        mean_description_chars=('description_char_len', 'mean'),
        std_description_chars=('description_char_len', 'std'),
        mean_description_tokens=('description_token_count', 'mean'),
        std_description_tokens=('description_token_count', 'std'),
        mean_description_ttr=('description_ttr', 'mean'),
        corpus_description_ttr=('description_tokens', corpus_ttr),
    )
    .reset_index()
)

word_features = word_features.merge(basic_desc_features, on='word_norm', how='left')
word_features.head()


## Description Embedding Features

Two pre-experiment semantic features:

- average similarity among the 100 golden descriptions for a word
- average similarity between the original word and its 100 golden descriptions

These use the existing embedding cache. Missing description embeddings are reported; if many are missing, rerun the model-comparison semantic analysis or run this notebook in an environment with `sentence-transformers`.


In [ ]:
description_texts = golden['description'].dropna().astype(str).unique().tolist()
word_texts = word_features['word_norm'].dropna().astype(str).unique().tolist()
desc_embeddings, missing_desc_embedding_texts = load_cached_embeddings(description_texts + word_texts)
print(f'Description/word embeddings found: {len(desc_embeddings):,}; missing: {len(missing_desc_embedding_texts):,}')

rows = []
for word, sub in golden.groupby('word_norm', observed=True):
    descs = [desc for desc in sub['description'].astype(str) if desc in desc_embeddings]
    word_vec = desc_embeddings.get(word)
    if not descs:
        rows.append({
            'word_norm': word,
            'golden_description_embedding_n': 0,
            'mean_pairwise_golden_description_similarity': np.nan,
            'mean_word_to_golden_description_similarity': np.nan,
        })
        continue
    vectors = np.vstack([desc_embeddings[desc] for desc in descs])
    sim = vectors @ vectors.T
    if len(descs) > 1:
        tri = sim[np.triu_indices(len(descs), k=1)]
        pairwise_mean = float(np.mean(tri))
    else:
        pairwise_mean = np.nan
    if word_vec is not None:
        word_desc_mean = float(np.mean(vectors @ word_vec))
    else:
        word_desc_mean = np.nan
    rows.append({
        'word_norm': word,
        'golden_description_embedding_n': len(descs),
        'mean_pairwise_golden_description_similarity': pairwise_mean,
        'mean_word_to_golden_description_similarity': word_desc_mean,
    })

description_embedding_features = pd.DataFrame(rows)
word_features = word_features.merge(description_embedding_features, on='word_norm', how='left')
pd.Series(missing_desc_embedding_texts, name='missing_text').to_csv(OUTPUT_DIR / 'missing_description_embedding_texts.csv', index=False)
word_features.head()


## Merge Modeling Dataset


In [ ]:
modeling = targets.merge(word_features, on=['category', 'original_word', 'word_norm'], how='left')
modeling['model_family'] = modeling['model_name'].map(lambda name: MODEL_META[name]['model_family'])
modeling['parameter_b'] = modeling['model_name'].map(lambda name: MODEL_META[name]['parameter_b'])
modeling['log_parameter_b'] = np.log10(modeling['parameter_b'])

modeling.to_csv(OUTPUT_DIR / 'regression_modeling_dataset.csv', index=False)
print(modeling.shape)
modeling.head()


## Train/Test Evaluation

The scikit-learn package is required for this section and should be installed in the analysis environment. The split is grouped by word, so the same word does not appear in both train and test with different models or pipelines.


## Improved Evaluation: Pruned Features + Grouped Model Search

Run this block after the dataset merge step. It keeps the predictors pre-experiment-only, drops constant/redundant features, adds a mean dummy baseline, and evaluates models with repeated grouped splits by `word_norm` so the same word does not appear in both train and validation folds.


In [ ]:
import sys
from pathlib import Path

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data_analysis.predicting_semantic_stability.regression_model_search import (
    ALL_CANDIDATE_FEATURES,
    SearchSettings,
    build_feature_sets,
    feature_diagnostics,
    run_grouped_model_search,
)

TARGET = 'step10_semantic_similarity'
SECONDARY_TARGET = 'step10_exact_accuracy'

FEATURE_SETS = build_feature_sets(NEIGHBOR_K)
SELECTED_FEATURE_SET = 'pruned_main'
feature_cols = [col for col in FEATURE_SETS[SELECTED_FEATURE_SET] if col in modeling.columns]

print(f'Selected feature set: {SELECTED_FEATURE_SET}')
print(f'Feature count: {len(feature_cols)}')
print(feature_cols)


In [ ]:
feature_report = feature_diagnostics(modeling, ALL_CANDIDATE_FEATURES, feature_cols)
feature_report.to_csv(OUTPUT_DIR / 'improved_feature_diagnostics.csv', index=False)
feature_report


### Grouped Hyperparameter Search

The validation splits are grouped by `word_norm`. This tests whether the model generalizes to held-out words rather than memorizing word-specific rows across model/pipeline conditions. The `mean_dummy` model predicts the training-fold mean and should be treated as the baseline floor.


In [ ]:
SEARCH_SETTINGS = SearchSettings(
    n_group_splits=10,
    group_test_size=0.20,
    n_random_search_iter=24,
    random_state=RANDOM_STATE,
)

progress = tqdm if 'tqdm' in globals() and tqdm else None

search_metrics, tuned_models, dropped_after_cleaning, search_info = run_grouped_model_search(
    modeling,
    TARGET,
    feature_cols,
    feature_set_name=SELECTED_FEATURE_SET,
    settings=SEARCH_SETTINGS,
    progress=progress,
)

search_metrics.to_csv(OUTPUT_DIR / f'improved_grouped_search_{TARGET}_{SELECTED_FEATURE_SET}.csv', index=False)
dropped_after_cleaning.to_csv(OUTPUT_DIR / f'improved_dropped_after_cleaning_{TARGET}_{SELECTED_FEATURE_SET}.csv', index=False)

print(search_info)
search_metrics


In [ ]:
exact_search_metrics, exact_tuned_models, exact_dropped_after_cleaning, exact_search_info = run_grouped_model_search(
    modeling,
    SECONDARY_TARGET,
    feature_cols,
    feature_set_name=SELECTED_FEATURE_SET,
    settings=SEARCH_SETTINGS,
    progress=progress,
)

exact_search_metrics.to_csv(OUTPUT_DIR / f'improved_grouped_search_{SECONDARY_TARGET}_{SELECTED_FEATURE_SET}.csv', index=False)
exact_dropped_after_cleaning.to_csv(OUTPUT_DIR / f'improved_dropped_after_cleaning_{SECONDARY_TARGET}_{SELECTED_FEATURE_SET}.csv', index=False)

print(exact_search_info)
exact_search_metrics


### Optional: Compare Feature Sets

This option is disabled by default because the grouped search is rerun for each feature set. It can be enabled after the main run to compare `category` one-hot encoding with explicit frequency and concreteness flags, or to test the explanatory value of model and pipeline metadata.


In [ ]:
RUN_FEATURE_SET_COMPARISON = True
FEATURE_SETS_TO_COMPARE = ['pruned_main', 'pruned_binary_category', 'model_pipeline_only', 'word_description_only']

if RUN_FEATURE_SET_COMPARISON:
    comparison_rows = []
    for feature_set_name in FEATURE_SETS_TO_COMPARE:
        cols = [col for col in FEATURE_SETS[feature_set_name] if col in modeling.columns]
        metrics_i, _, _, info_i = run_grouped_model_search(
            modeling,
            TARGET,
            cols,
            feature_set_name=feature_set_name,
            settings=SEARCH_SETTINGS,
            progress=progress,
        )
        metrics_i['feature_count'] = len(cols)
        metrics_i['n_rows'] = info_i['n_rows']
        metrics_i['n_groups'] = info_i['n_groups']
        comparison_rows.append(metrics_i)
    feature_set_comparison = pd.concat(comparison_rows, ignore_index=True).sort_values('cv_rmse_mean')
    feature_set_comparison.to_csv(OUTPUT_DIR / f'improved_feature_set_comparison_{TARGET}.csv', index=False)
    display(feature_set_comparison)
else:
    print('Set RUN_FEATURE_SET_COMPARISON = True to run the optional feature-set comparison.')
